In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 64.7 MB/s eta 0:00:00


In [4]:

import fitz  # PyMuPDF

def extract_cad_data(pdf_path):
    doc = fitz.open(pdf_path)
    dictionary_data = []

    for page in doc:
        # 1. Define the two columns (typical CAD page layout)
        width = page.rect.width
        height = page.rect.height
        mid = width / 2

        # Left and Right column bounding boxes
        columns = [
            fitz.Rect(0, 0, mid, height),
            fitz.Rect(mid, 0, width, height)
        ]

        for col in columns:
            blocks = page.get_text("dict", clip=col)["blocks"]

            for b in blocks:
                if "lines" in b:
                    for line in b["lines"]:
                        for span in line["spans"]:
                            text = span["text"].strip()
                            font_flags = span["flags"] # Contains bold info

                            # Bit 4 of flags usually indicates Bold
                            is_bold = font_flags & 2**4

                            if is_bold and len(text) > 1:
                                # This is likely a headword
                                dictionary_data.append({
                                    "headword": text,
                                    "content": ""
                                })
                            elif dictionary_data:
                                # Append to the most recent headword's definition
                                dictionary_data[-1]["content"] += " " + text

    return dictionary_data



In [5]:
data = extract_cad_data("/content/drive/MyDrive/CAD_Volume_A1.pdf")

In [6]:
data

[{'headword': 'ASSYRIAN', 'content': ' D OF THE ORIENTAL INSTITUTE OF'},
 {'headword': 'EDITORIAL', 'content': ' IGNACEJ.'},
 {'headword': 'GELB,', 'content': ' BENNO LANDSBERGER ------ 196 PUBLISHED'},
 {'headword': 'BY', 'content': ''},
 {'headword': 'THE',
  'content': ' ORIENTAL INSTI AND J.J. AUGUSTIN VERLAGSBUCHHAN oi.uchic HE'},
 {'headword': 'DICTIONARY', 'content': ' F THE UNIVERSITY OF CHICAGO L'},
 {'headword': 'BOARD', 'content': ' R, A.'},
 {'headword': 'LEO', 'content': ''},
 {'headword': 'OPPENHEIM,',
  'content': ' ERICA REINER 64 ITUTE, CHICAGO, ILLINOIS, U.S.A. NDLUNG, GLOCKSTADT, GERMANY cago.edu INTERNATIONAL STANDARD BO (SET: 0-918 LIBRARY OF CONGRESS CATAL COPYRIGHT UNDER THE INTERNAT ALL RIGHTS R THE ORIENTAL INSTITUT Fourth Prin PRINTED IN THE UNITED COMPOSITION BY J. J. A oi.uchic OOK NUMBER: 0-918986-06-0 8986-05-2) LOG CARD NUMBER: 56-58292 TIONAL COPYRIGHT UNION, 1964 RESERVED by TE, CHICAGO, ILLINOIS nting 1998 D STATES OF AMERICA AUGUSTIN, GLUCKSTADT cago.

In [7]:
import re

# Example string from the dictionary:
# "šarru bēlīya the king, my lord (ABL 123)"

def separate_languages(text):
    # This regex looks for text often found inside English translations
    # or identifies the 'Akkadian' part by looking for italicized-style diacritics
    # and then splitting where common English words start.

    # Simple split example: Looking for the citation at the end (e.g., ABL 123)
    parts = re.split(r'(\([A-Z]+\s\d+\))', text)

    return parts

# Once your PDF script works, you'll pass the 'content' through a filter like this.

In [8]:
import fitz

def extract_with_language_tags(pdf_path):
    doc = fitz.open(pdf_path)
    entries = []

    for page in doc:
        # Standard CAD dual-column split logic
        mid = page.rect.width / 2
        columns = [fitz.Rect(0, 0, mid, page.rect.height),
                   fitz.Rect(mid, 0, page.rect.width, page.rect.height)]

        for col in columns:
            blocks = page.get_text("dict", clip=col)["blocks"]
            for b in blocks:
                if "lines" not in b: continue

                for line in b["lines"]:
                    for span in line["spans"]:
                        text = span["text"].strip()
                        if not text: continue

                        flags = span["flags"]
                        is_italic = flags & 2**1  # Bit 1 is Italic
                        is_bold = flags & 2**4    # Bit 4 is Bold

                        if is_bold:
                            entries.append({"headword": text, "examples": []})
                        elif is_italic:
                            # Start a new example pair or add to existing
                            if entries:
                                entries[-1]["examples"].append({"akk": text, "eng": ""})
                        else:
                            # Normal text is usually the English translation
                            if entries and entries[-1]["examples"]:
                                entries[-1]["examples"][-1]["eng"] += " " + text

    return entries

In [9]:
data2 = extract_with_language_tags("/content/drive/MyDrive/CAD_Volume_A1.pdf")

In [10]:
import re

def clean_dictionary_data(raw_entries):
    clean_data = []

    # Pattern to catch headers like "A/1" or "šarru — 104"
    # and footers/page numbers
    header_pattern = re.compile(r'^[A-Z]/?\d*|—\s\d+$|Page\s\d+')

    for entry in raw_entries:
        # 1. Clean the headword
        headword = entry['headword'].strip()
        if header_pattern.match(headword) or len(headword) < 2:
            continue

        clean_examples = []
        for ex in entry['examples']:
            # 2. Fix hyphenation (e.g., "šar- \n ru" -> "šarru")
            akk = re.sub(r'-\s+', '', ex['akk']).strip()
            eng = re.sub(r'-\s+', '', ex['eng']).strip()

            # 3. Basic Validation:
            # Ensure the Akkadian part isn't just a number or reference
            if not any(char.isalpha() for char in akk):
                continue

            # Ensure we actually have an English translation for the pair
            if len(eng) > 3:
                clean_examples.append({"akk": akk, "eng": eng})

        if clean_examples:
            clean_data.append({
                "headword": headword,
                "examples": clean_examples
            })

    return clean_data

In [12]:
data2

[{'headword': 'ASSYRIAN', 'examples': []},
 {'headword': 'D', 'examples': []},
 {'headword': 'EDITORIAL', 'examples': []},
 {'headword': 'GELB,',
  'examples': [{'akk': '------', 'eng': ' 196 PUBLISHED'}]},
 {'headword': 'BY', 'examples': []},
 {'headword': 'THE', 'examples': []},
 {'headword': 'DICTIONARY', 'examples': []},
 {'headword': 'L', 'examples': []},
 {'headword': 'BOARD', 'examples': []},
 {'headword': 'LEO', 'examples': []},
 {'headword': 'OPPENHEIM,',
  'examples': [{'akk': 'Fourth', 'eng': ''},
   {'akk': 'Prin',
    'eng': ' PRINTED IN THE UNITED COMPOSITION BY J. J. A oi.uchic OOK NUMBER: 0-918986-06-0 8986-05-2) LOG CARD NUMBER: 56-58292 TIONAL COPYRIGHT UNION, 1964 RESERVED'},
   {'akk': 'by', 'eng': ' TE, CHICAGO, ILLINOIS'},
   {'akk': 'nting',
    'eng': ' 1998 D STATES OF AMERICA AUGUSTIN, GLUCKSTADT cago.edu THE ASSYRIAN VOLU'}]},
 {'headword': 'A', 'examples': []},
 {'headword': 'PAR', 'examples': []},
 {'headword': 'A.', 'examples': []},
 {'headword': 'LEO', 'e